# Build Team Ops Index Once (Save To Drive)

Use this notebook once to create embeddings and persist the index to Google Drive.
Then use the demo notebook to query without rebuilding embeddings.

## 1) Enable GPU In Colab

- Runtime -> Change runtime type
- Hardware accelerator -> T4 GPU
- Save

In [ ]:
# Install Python packages
%pip install -q jedi pypdf llama-index llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
# Install system dependencies and Ollama
!apt-get -qq update && apt-get -qq install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama in background
import subprocess, time, os
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
print('Ollama started.')

In [ ]:
# Pull generation and embedding models
!ollama pull llama3.2:3b
!ollama pull nomic-embed-text

## 2) Mount Drive And Extract Operations ZIP

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path('/content/drive/MyDrive/Operations.zip')
data_dir = Path('/content/team_ops_data')

if not zip_path.exists():
    raise FileNotFoundError(f'Missing ZIP: {zip_path}')

data_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(data_dir)

print(f'Extracted to {data_dir}')

## 3) Build Embeddings + Persist Index To Drive

This is the slow step. Run once, then reuse the saved index.

In [ ]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model='llama3.2:3b', request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name='nomic-embed-text')
Settings.chunk_size = 512
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader(str(data_dir), recursive=True).load_data()
print(f'Loaded {len(documents)} docs')

index = VectorStoreIndex.from_documents(documents)

persist_dir = '/content/drive/MyDrive/team_ops_index_nomic'
index.storage_context.persist(persist_dir=persist_dir)
print(f'Persisted index to: {persist_dir}')

In [ ]:
# Optional quick validation query
qe = index.as_query_engine(similarity_top_k=4)
print(qe.query('How do I submit my timesheets?'))